In [ ]:
# For Gpu setup
!nvidia-smi

In [ ]:
# check gpu is working
import torch
print(torch.cuda.is_available())

In [ ]:
# Import the files of roboflow
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="8RRAkqmXcuvEFawT2jdA")
project = rf.workspace("palaks-workspace-bt0bl").project("ai-archaeological-site")
version = project.version(1)
dataset = version.download("yolov8")

In [ ]:
!ls /content/AI-Archaeological-Site-1/train

In [ ]:
train_path = "/content/AI-Archaeological-Site-1/train"
valid_path = "/content/AI-Archaeological-Site-1/valid"

In [ ]:
train_path = "/content/AI-Archaeological-Site-1/train"
valid_path = "/content/AI-Archaeological-Site-1/valid"

!ls {train_path}
!ls {valid_path}

In [ ]:
# Generate polygon masks
import os
import cv2
import numpy as np

def generate_masks_polygon(img_dir, label_dir, mask_dir):
    os.makedirs(mask_dir, exist_ok=True)  # create folder if missing
    for file in os.listdir(img_dir):
        if not file.endswith(".jpg"):
            continue
        img_path = os.path.join(img_dir, file)
        label_path = os.path.join(label_dir, file.replace(".jpg", ".txt"))
        if not os.path.exists(label_path):
            continue

        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)

        with open(label_path) as f:
            for line in f.readlines():
                values = list(map(float, line.strip().split()))
                cls = int(values[0])
                coords = values[1:]
                pts = np.array(coords).reshape(-1, 2)
                pts[:,0] *= w
                pts[:,1] *= h
                pts = pts.astype(np.int32)
                cv2.fillPoly(mask, [pts], 255)

        cv2.imwrite(os.path.join(mask_dir, file), mask)

# Run for train & valid
train_path = "/content/AI-Archaeological-Site-1/train"
valid_path = "/content/AI-Archaeological-Site-1/valid"

generate_masks_polygon(train_path+"/images", train_path+"/labels", train_path+"/masks")
generate_masks_polygon(valid_path+"/images", valid_path+"/labels", valid_path+"/masks")

# Verify masks
!ls {train_path}/masks | head
!ls {valid_path}/masks | head

In [ ]:
#U-net dataset loader
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import os

class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.files = os.listdir(img_dir)
        self.transform = T.Compose([
            T.Resize((256,256)),
            T.ToTensor()
        ])
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        file = self.files[idx]
        img = Image.open(os.path.join(self.img_dir, file)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, file)).convert("L")
        img = self.transform(img)
        mask = self.transform(mask)
        return img, mask

train_dataset = SegDataset(train_path+"/images", train_path+"/masks")
val_dataset = SegDataset(valid_path+"/images", valid_path+"/masks")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [ ]:
#U-net model, apply loss and optimizer
!pip install segmentation-models-pytorch

import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)
model.to(device)

loss_fn = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# U-net training loop + IoU/Dice Validation
from tqdm import tqdm

def dice_coef(pred, target, smooth=1.):
    pred = (pred > 0.5).float()
    target = target.float()
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

epochs = 10  # for testing, baad me increase
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for imgs, masks in tqdm(train_loader):
        imgs = imgs.to(device)
        masks = masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = loss_fn(preds, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    print(f"Epoch {epoch+1}, Train Loss: {train_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    dice_score = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            preds = model(imgs)
            dice_score += dice_coef(preds, masks).item()
    print(f"Epoch {epoch+1}, Validation Dice: {dice_score/len(val_loader):.4f}")

In [ ]:
# Calculate Dice/IoU metric function
import torch

def dice_coef(pred, target, smooth=1.0):
    """
    Compute Dice coefficient
    pred: predicted mask (B,C,H,W) tensor
    target: ground truth mask (B,C,H,W) tensor
    """
    pred = (pred > 0.5).float()
    target = target.float()
    intersection = (pred * target).sum(dim=(1,2,3))
    dice = (2. * intersection + smooth) / (pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + smooth)
    return dice.mean()

In [ ]:
# U-net training function and return the average loss
def train_unet(model, train_loader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0
    for imgs, masks in train_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        preds = model(imgs)
        loss = loss_fn(preds, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    return running_loss / len(train_loader)

In [ ]:
# Add the IoU function
def iou_coef(pred, target, smooth=1.0):
    """
    Compute Intersection over Union (IoU)
    pred: predicted mask (B,C,H,W)
    target: ground truth mask (B,C,H,W)
    """
    pred = (pred > 0.5).float()
    target = target.float()

    intersection = (pred * target).sum(dim=(1,2,3))
    union = (pred + target - pred*target).sum(dim=(1,2,3))
    iou = (intersection + smooth) / (union + smooth)
    return iou.mean()

In [ ]:
# Validate function and return the Dicescore and IoU
def validate_unet(model, val_loader, device):
    model.eval()
    dice_score = 0
    iou_score = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            preds = model(imgs)

            dice_score += dice_coef(preds, masks).item()
            iou_score += iou_coef(preds, masks).item()

    dice_avg = dice_score / len(val_loader)
    iou_avg = iou_score / len(val_loader)
    return dice_avg, iou_avg

In [ ]:
# Training + Validation loop
from tqdm import tqdm
epochs = 10

for epoch in range(epochs):
    train_loss = train_unet(model, train_loader, optimizer, loss_fn, device)
    val_dice, val_iou = validate_unet(model, val_loader, device)
    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Validation Dice = {val_dice:.4f}, IoU = {val_iou:.4f}")

In [ ]:
# Save the U-net model path
torch.save(model.state_dict(), "/content/u-net_model.pth")

In [ ]:
# Check the path of data.yaml file
!cat /content/AI-Archaeological-Site-1/data.yaml

In [ ]:
# Correct the path of data.yaml file
data_yaml = """
train: /content/AI-Archaeological-Site-1/train/images
val: /content/AI-Archaeological-Site-1/valid/images
nc: 1
names: ['objects-ruins-artifacts']
"""

with open("/content/AI-Archaeological-Site-1/data.yaml", "w") as f:
    f.write(data_yaml)

In [ ]:
# Install the yolov8 model
!pip install ultralytics --upgrade

In [ ]:
# YOLOv8 Segmentation training
!yolo task=segment mode=train model=yolov8s-seg.pt data=/content/AI-Archaeological-Site-1/data.yaml epochs=50 imgsz=640

In [ ]:
# YOLOv8 model evaluation
!yolo task=segment mode=val model=runs/segment/train/weights/best.pt data=/content/AI-Archaeological-Site-1/data.yaml

In [ ]:
# YOLOv8 object detection
!yolo task=detect mode=train model=yolov8s.pt data=/content/AI-Archaeological-Site-1/data.yaml epochs=50 imgsz=640

# Predict the terrain erosion model

In [ ]:
# Check the path of train images
train_img_path = "/content/AI-Archaeological-Site-1/train/images"

In [ ]:
# Define the images
import os

images = os.listdir(train_img_path)
print("Total images:", len(images))

In [ ]:
# Check for the masks
mask_folder = "/content/AI-Archaeological-Site-1/train/masks"

In [ ]:
# Feature extraction function
import cv2
import numpy as np

def extract_features(image_path, mask_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 🔹 slope (fake)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1)
    slope = np.mean(np.sqrt(sobelx**2 + sobely**2))

    # 🔹 vegetation index
    green = img[:,:,1]
    red = img[:,:,2]
    veg_index = np.mean((green - red) / (green + red + 1e-6))

    # 🔹 texture
    texture = np.var(gray)

    # 🔹 mask feature
    mask = cv2.imread(mask_path, 0)

    # mask values check karo (ek baar)
    # print(np.unique(mask))

    veg_area = np.sum(mask == 255)   # agar binary mask hai

    return [slope, veg_index, texture, veg_area]

In [ ]:
# Make the dataset (X,Y)
import os

X = []
y = []

for img_name in images:
    img_path = os.path.join(train_img_path, img_name)
    mask_path = os.path.join(mask_folder, img_name)

    # check file exist
    if not os.path.exists(mask_path):
        continue

    features = extract_features(img_path, mask_path)
    X.append(features)

In [ ]:
# Use precentile-based thresold for erosion
import numpy as np

X_array = np.array(X)

# Slope threshold → top 25% high gradient
slope_thresh = np.percentile(X_array[:,0], 75)

# Vegetation threshold → bottom 25% low vegetation
veg_thresh = np.percentile(X_array[:,1], 25)

y_auto = []
for f in X_array:
    slope_score = f[0]
    veg_score = f[1]

    # Rule: high slope OR low vegetation → erosion prone
    if slope_score > slope_thresh or veg_score < veg_thresh:
        y_auto.append(1)  # erosion prone
    else:
        y_auto.append(0)  # stable

y = np.array(y_auto)

In [ ]:
# Check the labels
print("Erosion prone:", np.sum(y==1))
print("Stable:", np.sum(y==0))

In [ ]:
# Verify the slope+vegetation index rule prediction
import matplotlib.pyplot as plt
import cv2

sample_idx = np.random.randint(0, len(images), 5)
for i in sample_idx:
    img_path = os.path.join(train_img_path, images[i])
    img = cv2.imread(img_path)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Label: {y[i]}")
    plt.show()

In [ ]:
# Convert to the numpy
import numpy as np

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)

In [ ]:
# Train-Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train the XGBoost
!pip install xgboost
from xgboost import XGBClassifier

model = XGBClassifier(n_estimators=100, max_depth=5)
model.fit(X_train, y_train)

In [ ]:
# For save the erosion model
model.save_model("erosion_model.json")

In [ ]:
# Download the erosion model
from google.colab import files
files.download("erosion_model.json")

In [ ]:
# Evaluate the model
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import numpy as np

y_pred = model.predict(X_test)

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# Feature Importance
import matplotlib.pyplot as plt

plt.bar(["slope","veg_index","texture","veg_area"], model.feature_importances_)
plt.title("Feature Importance")
plt.show()

In [ ]:
# Check veg_area distribution
import matplotlib.pyplot as plt

veg_area = X[:,3]  # veg_area column
plt.hist(veg_area, bins=20)
plt.title("veg_area distribution")
plt.show()

print("Min:", veg_area.min(), "Max:", veg_area.max(), "Mean:", veg_area.mean())

In [ ]:
# Prepare the Image Coordinates
import pandas as pd
import numpy as np

# Number of images
n = len(images)  # images list from your dataset

# Create dummy coordinates (for visualization)
lats = np.linspace(28.7040, 28.7055, n)
lons = np.linspace(77.1010, 77.1035, n)

# Create dataframe
df = pd.DataFrame({
    "image_name": images,
    "lat": lats,
    "lon": lons,
    "label": y  # 0 = stable, 1 = erosion-prone
})

print("Sample dataframe:")
print(df.head())

In [ ]:
# Install the folium for map plotting
!pip install folium

In [ ]:
# Create basic scatter map
import folium

# Center map on mean coordinates
map_center = [df["lat"].mean(), df["lon"].mean()]
m = folium.Map(location=map_center, zoom_start=16)

# Plot each image as a circle
for idx, row in df.iterrows():
    color = "red" if row["label"] == 1 else "green"
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=row["image_name"]
    ).add_to(m)

# Display map
m
# Red dots- Erosion-prone, Green dots- Stable